# Reference images via image-conditioned generation (SDXL + IP-Adapter, style-only)

The **target photo itself is fed to the generator** as an image prompt. IP-Adapter is restricted to the *style* layers of SDXL, so it passes on colour, material, texture and look, while the text prompt names a **different object** that sets the content/geometry.

- image input -> appearance, material, style
- text input (`a photo of a <other object>`) -> what is drawn (different geometry)
- negative prompt = the target's own category, so the target object is not redrawn
- DINOv2 check flags references that are still too similar to the target and regenerates them once

Input: `sketchy_captions_sample.csv`. Needs a GPU runtime (T4 is fine, roughly 10-15 s per image).

In [1]:
!pip install -q -U diffusers transformers accelerate

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os, gc, torch, numpy as np, pandas as pd
from PIL import Image
from tqdm.auto import tqdm

# ---------------- CONFIG ----------------
ROOT = '/content/drive/MyDrive/sketchy_dataset_one_sketch'
PAIRS_CSV = os.path.join(ROOT, 'sketchy_pairs.csv')            # full list -> pool of 'other' categories
CSV_IN = os.path.join(ROOT, 'sketchy_captions_sample.csv')     # the 200 captioned rows
CSV_OUT = os.path.join(ROOT, 'sketchy_with_refs.csv')
REF_DIR = os.path.join(ROOT, 'reference_images')

SDXL_ID = 'stabilityai/stable-diffusion-xl-base-1.0'
RES, STEPS, CFG = 768, 20, 5.0

STYLE_ONLY = True      # True: IP-Adapter only in SDXL's style block (least geometry leakage)
FULL_SCALE = 0.6       # used when STYLE_ONLY=False (whole adapter, 0-1; higher = copies the target more, incl. geometry)

T2I_PROMPT = 'a photo of a {obj}'
SIM_THRESH = 0.5       # DINOv2 cosine above this = 'too similar to target' (tune after looking at the stats)
SEED = 0
# ----------------------------------------

df = pd.read_csv(CSV_IN).reset_index(drop=True)
all_cats = sorted(pd.read_csv(PAIRS_CSV)['category'].unique())
print(len(df), 'rows |', len(all_cats), 'categories in pool')

200 rows | 125 categories in pool


In [4]:
from diffusers import AutoPipelineForText2Image
from transformers import CLIPVisionModelWithProjection
import torch
import gc

# 1. Clear GPU and system memory
if 'pipe' in locals():
    del pipe
if 'image_encoder' in locals():
    del image_encoder

gc.collect()
torch.cuda.empty_cache()

# 2. Load the suggested matching IP-Adapter image encoder
# Using low_cpu_mem_usage=True to prevent system RAM crashes
image_encoder = CLIPVisionModelWithProjection.from_pretrained(
    "h94/IP-Adapter",
    subfolder="models/image_encoder",
    torch_dtype=torch.float16,
    low_cpu_mem_usage=True
).to("cuda")

# 3. Load SDXL base with CPU offload & low memory usage enabled
pipe = AutoPipelineForText2Image.from_pretrained(
    SDXL_ID,
    torch_dtype=torch.float16,
    variant='fp16',
    image_encoder=image_encoder,
    low_cpu_mem_usage=True
)
pipe.enable_model_cpu_offload()  # Manages GPU VRAM dynamically

# 4. Load the matching IP-Adapter
pipe.load_ip_adapter('h94/IP-Adapter', subfolder='sdxl_models', weight_name='ip-adapter_sdxl_vit-h.bin')
pipe.set_progress_bar_config(disable=True)

if STYLE_ONLY:
    pipe.set_ip_adapter_scale({'up': {'block_0': [0.0, 1.0, 0.0]}})   # style-only injection
else:
    pipe.set_ip_adapter_scale(FULL_SCALE)

[transformers] `Siglip2ImageProcessorFast` is deprecated. The `Fast` suffix for image processors has been removed; use `Siglip2ImageProcessor` instead.


config.json:   0%|          | 0.00/560 [00:00<?, ?B/s]

models/image_encoder/model.safetensors: reconstructing file:   0%|          |  0.00B / 2.53GB            

models/image_encoder/model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/520 [00:00<?, ?it/s]

...

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

sdxl_models/ip-adapter_sdxl_vit-h.bin: reconstructing file:   0%|          |  0.00B /  698MB            

sdxl_models/ip-adapter_sdxl_vit-h.bin: downloading bytes:           |  0.00B            

There are modules in UNet2DConditionModel that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.


In [ ]:
import numpy as np
import torch
import os
from PIL import Image

rng = np.random.default_rng(SEED)
def pick_other(cat):
    return str(rng.choice([c for c in all_cats if c != cat]))

def ref_path_for(row):
    stem = os.path.splitext(os.path.basename(row.image_path))[0]
    d = os.path.join(REF_DIR, row.category); os.makedirs(d, exist_ok=True)
    return os.path.join(d, f'{stem}_ref.png')

@torch.no_grad()
def make_ref(img_path, ref_cat, tgt_cat, seed):
    img = Image.open(img_path).convert('RGB')           # <- the target photo goes into the generator
    return pipe(
        prompt=T2I_PROMPT.format(obj=ref_cat.replace('_', ' ')),
        negative_prompt=f"{tgt_cat.replace('_', ' ')}, blurry, low quality, deformed",
        ip_adapter_image=img,
        num_inference_steps=STEPS, guidance_scale=CFG, height=RES, width=RES,
        generator=torch.Generator('cuda').manual_seed(int(seed)),
    ).images[0]

df['ref_category'] = [pick_other(c) for c in df.category]
df['ref_path'] = [ref_path_for(r) for r in df.itertuples()]

# quick test on 3 images before the full run
for i in range(3):
    r = df.iloc[i]
    make_ref(r.image_path, r.ref_category, r.category, SEED + i).save(r.ref_path)
    print(i, r.category, '->', r.ref_category)

/usr/local/lib/python3.13/dist-packages/diffusers/pipelines/stable_diffusion_xl/pipeline_stable_diffusion_xl.py:748: FutureWarning: `upcast_vae` is deprecated and will be removed in version 1.0.0. `upcast_vae` is deprecated. Please use `pipe.vae.to(torch.float32)`. For more details, please refer to: https://github.com/huggingface/diffusers/pull/12619#issue-3606633695.
  deprecate(
There are modules in AutoencoderKL that should be kept in float32: []. Casting directly with `to()` can lead to inconsistent results; set `torch_dtype` in `from_pretrained()` instead to keep these modules in float32.


In [ ]:
# full run (resumable: skips references that already exist on Drive)
for i, r in tqdm(list(df.iterrows())):
    if os.path.exists(r.ref_path):
        continue
    make_ref(r.image_path, r.ref_category, r.category, SEED + i).save(r.ref_path)

df.to_csv(CSV_OUT, index=False)
print('done')

## Check that references are not too close to their targets

In [ ]:
from transformers import AutoModel, AutoImageProcessor

dproc = AutoImageProcessor.from_pretrained('facebook/dinov2-small')
dino = AutoModel.from_pretrained('facebook/dinov2-small').to('cuda').eval()

@torch.no_grad()
def dino_emb(path):
    x = dproc(images=Image.open(path).convert('RGB'), return_tensors='pt').to('cuda')
    return torch.nn.functional.normalize(dino(**x).last_hidden_state[:, 0], dim=-1)[0]

def sim(a, b):
    return float(dino_emb(a) @ dino_emb(b))

df['dino_sim'] = [sim(r.image_path, r.ref_path) for r in df.itertuples()]
print(df.dino_sim.describe())
print('above threshold:', int((df.dino_sim > SIM_THRESH).sum()))

In [ ]:
# one regeneration pass for refs that are too similar (new random object + new seed)
bad = df.index[df.dino_sim > SIM_THRESH]
for i in tqdm(bad):
    r = df.loc[i]
    new_cat = pick_other(r.category)
    make_ref(r.image_path, new_cat, r.category, SEED + 1000 + i).save(r.ref_path)
    df.loc[i, 'ref_category'] = new_cat
    df.loc[i, 'dino_sim'] = sim(r.image_path, r.ref_path)

print('still above threshold:', int((df.dino_sim > SIM_THRESH).sum()))
df.to_csv(CSV_OUT, index=False)
print('Saved ->', CSV_OUT)

In [ ]:
import io, base64
from IPython.display import HTML, display

def b64(path, size=220):
    im = Image.open(path).convert('RGB'); im.thumbnail((size, size))
    buf = io.BytesIO(); im.save(buf, format='JPEG', quality=80)
    return base64.b64encode(buf.getvalue()).decode()

def show(start=0, n=12):
    rows = ''
    for i, r in df.iloc[start:start + n].iterrows():
        rows += (f"<tr><td style='padding:4px'>{i}</td>"
                 f"<td><img src='data:image/jpeg;base64,{b64(r.image_path)}'></td>"
                 f"<td><img src='data:image/jpeg;base64,{b64(r.ref_path)}'></td>"
                 f"<td style='padding:6px'><b>target:</b> {r.category}<br>"
                 f"<b>ref object:</b> {r.ref_category}<br><b>dino_sim:</b> {r.dino_sim:.2f}</td></tr>")
    display(HTML(f'<table>{rows}</table>'))

show(0, 12)   # page through with show(12, 12), show(24, 12), ...